# Trying Traditional ML models 

# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 3: Evaluation, Baselines, Traditional ML

Today we'll write some simple models to predict the price of a product

We'll use an approach to evaluate the performance of the model

And we'll test some Baseline Models using Traditional machine learning

In [2]:
import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from pricer.evaluator import evaluate
from pricer.items import Item


In [3]:
LITE_MODE = True

In [5]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"
train ,val ,test = Item.from_hub(dataset)

README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [6]:
print(f"Train :{len(train)}")
print(f"Val :{len(val)}")
print(f"Test :{len(test)}")





Train :20000
Val :1000
Test :1000


In [9]:
def random_pricer(item):
    return random.randrange(1,100)

In [10]:
random.seed(42)
evaluate(random_pricer,test)

  0%|          | 0/200 [00:00<?, ?it/s]

$137 $101 $51 $25 $6 $198 $115 $197 $59 $316 $626 $284 $50 $42 $3 $22 $74 $46 $198 $91 $104 $59 $7 $171 $260 $427 $403 $64 $9 $10 $34 $31 $102 $4 $79 $818 $38 $35 $124 $17 $156 $44 $15 $227 $122 $19 $4 $20 $96 $189 $11 $85 $397 $86 $47 $40 $52 $29 $186 $37 $225 $7 $53 $29 $549 $43 $166 $420 $34 $67 $26 $68 $250 $80 $12 $25 $94 $3 $32 $17 $61 $70 $17 $78 $21 $14 $155 $220 $85 $44 $66 $12 $10 $43 $74 $206 $35 $63 $238 $554 $40 $68 $2 $57 $160 $46 $4 $312 $11 $200 $52 $284 $26 $87 $144 $135 $16 $8 $17 $11 $25 $61 $14 $102 $149 $23 $39 $80 $145 $121 $39 $41 $13 $39 $31 $81 $130 $15 $53 $86 $17 $221 $62 $46 $130 $20 $52 $503 $250 $13 $62 $85 $41 $85 $58 $270 $1 $7 $243 $10 $221 $7 $29 $11 $752 $70 $433 $38 $29 $87 $10 $77 $337 $21 $87 $111 $28 $72 $35 $92 $561 $68 $266 $184 $48 $1 $68 $53 $21 $12 $47 $14 $43 $113 $80 $32 $72 $259 $19 $42 

In [11]:
prices = [item.price for item in train]
avg_price = sum(prices) / len(prices)
def avg_pricer(item):
    return avg_price
evaluate(avg_pricer,test)




  0%|          | 0/200 [00:00<?, ?it/s]

$79 $24 $85 $70 $110 $90 $4 $75 $105 $190 $573 $239 $120 $86 $61 $107 $61 $90 $70 $21 $6 $16 $55 $35 $192 $312 $355 $120 $42 $60 $120 $80 $20 $60 $25 $679 $81 $84 $73 $102 $59 $60 $105 $115 $80 $115 $122 $109 $5 $62 $105 $10 $335 $21 $87 $6 $133 $100 $62 $128 $96 $62 $49 $30 $488 $50 $100 $305 $15 $64 $109 $123 $140 $121 $90 $104 $16 $130 $123 $122 $20 $128 $110 $41 $113 $80 $42 $166 $20 $95 $118 $45 $120 $105 $132 $88 $106 $17 $130 $435 $40 $23 $103 $1 $109 $22 $115 $260 $109 $159 $80 $174 $109 $12 $55 $30 $116 $120 $84 $37 $124 $51 $70 $26 $60 $80 $120 $41 $39 $1 $69 $3 $55 $110 $75 $125 $65 $70 $12 $2 $76 $110 $60 $121 $54 $108 $95 $370 $125 $107 $121 $34 $93 $0 $121 $139 $91 $84 $179 $90 $149 $114 $98 $127 $700 $117 $307 $90 $100 $130 $115 $118 $280 $118 $39 $9 $112 $47 $46 $47 $514 $115 $160 $108 $90 $118 $7 $73 $80 $114 $105 $89 $105 $2 $40 $60 $30 $140 $89 $114 

In [14]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight is None else 0,
        "length": len(item.summary)
}

In [15]:
def list_dataFrame(items):
    features =[get_features(item) for item in items]
    data_frame = pd.DataFrame(features)
    data_frame['price']=[item.price for item in items]
    return data_frame
train_df = list_dataFrame(train)
test_df = list_dataFrame(test)

In [17]:
feature_columns = ['weight', 'weight_unknown', 'length']
X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

model = LinearRegression()
model.fit(X_train,y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test,y_pred)
r2 = r2_score(y_test,y_pred)
print(f"MSE: {mse}")
print(f"R2: {r2}")




weight: 3.504043126937085
weight_unknown: 0.0
length: 0.2088932681056724
Intercept: 42.29184250999698
MSE: 20009.946612638025
R2: 0.16454177920137958


In [18]:
def linear_pricer(item):
    features = get_features(item)
    df = pd.DataFrame([features])
    return model.predict(df)[0]


In [19]:
evaluate(linear_pricer,test)

  0%|          | 0/200 [00:00<?, ?it/s]

$75 $53 $66 $70 $74 $50 $6 $48 $85 $209 $570 $230 $117 $61 $31 $101 $54 $65 $40 $11 $21 $11 $78 $7 $190 $314 $355 $94 $37 $39 $86 $60 $24 $19 $81 $694 $68 $70 $66 $70 $61 $39 $64 $64 $101 $97 $86 $87 $21 $57 $88 $23 $345 $5 $82 $38 $113 $87 $20 $105 $108 $53 $22 $0 $243 $20 $79 $258 $11 $72 $88 $87 $152 $92 $94 $83 $21 $101 $102 $91 $73 $89 $88 $6 $87 $46 $78 $173 $64 $63 $91 $25 $83 $74 $94 $115 $81 $10 $164 $415 $35 $9 $110 $18 $118 $4 $86 $295 $86 $195 $66 $152 $110 $61 $68 $50 $85 $97 $86 $16 $102 $47 $55 $48 $126 $50 $89 $104 $73 $13 $57 $27 $50 $75 $48 $87 $78 $43 $6 $13 $42 $129 $38 $88 $5 $74 $74 $382 $4 $74 $97 $29 $56 $18 $93 $116 $106 $43 $42 $61 $156 $95 $70 $98 $722 $103 $276 $78 $87 $104 $89 $103 $242 $81 $140 $60 $98 $10 $53 $41 $290 $120 $37 $94 $78 $99 $3 $60 $60 $82 $82 $82 $112 $18 $17 $99 $47 $107 $94 $97 

In [26]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]
vectorizer = CountVectorizer(max_features=1000,stop_words='english')
X = vectorizer.fit_transform(documents)


In [27]:
selected_features = vectorizer.get_feature_names_out()

print(f"len(selected_features): {len(selected_features)}")

print(selected_features[1000:1110])

len(selected_features): 1000
[]


In [28]:
regressor = LinearRegression()
regressor.fit(X,prices)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [29]:
def natualLanguagePricer(item):
    features = vectorizer.transform([item.summary])
    return max(regressor.predict(features)[0],0)






In [30]:
evaluate(natualLanguagePricer,test)

  0%|          | 0/200 [00:00<?, ?it/s]

$70 $97 $55 $58 $112 $223 $2 $109 $54 $39 $488 $187 $59 $171 $22 $47 $64 $19 $10 $63 $49 $82 $9 $42 $284 $314 $113 $41 $49 $36 $78 $73 $4 $30 $76 $437 $37 $86 $124 $80 $129 $107 $40 $23 $75 $34 $77 $15 $42 $13 $49 $59 $228 $89 $108 $61 $101 $186 $7 $12 $14 $32 $14 $37 $439 $77 $30 $171 $55 $225 $7 $61 $92 $105 $36 $62 $71 $53 $30 $128 $105 $45 $57 $45 $8 $70 $6 $176 $79 $113 $15 $164 $85 $43 $2 $102 $64 $3 $155 $122 $77 $91 $12 $15 $69 $9 $67 $153 $31 $112 $59 $15 $58 $4 $73 $60 $106 $21 $57 $34 $11 $110 $57 $18 $72 $51 $20 $142 $17 $45 $39 $111 $117 $9 $63 $38 $5 $109 $87 $58 $87 $121 $8 $58 $97 $55 $40 $309 $25 $56 $19 $247 $2 $33 $84 $113 $179 $25 $1 $31 $46 $26 $42 $13 $385 $15 $30 $50 $14 $38 $1 $61 $321 $52 $17 $14 $28 $49 $37 $125 $411 $23 $91 $41 $135 $60 $51 $2 $125 $10 $78 $78 $16 $51 $31 $84 $14 $28 $34 $11 

## Random Forest model

The Random Forest is a type of "**ensemble**" algorithm, meaning that it combines many smaller algorithms to make better predictions.

It uses a very simple kind of machine learning algorithm called a **decision tree**. A decision tree makes predictions by examining the values of features in the input. Like a flow chart with IF statements. Decision trees are very quick and simple, but they tend to overfit.

In our case, the "features" are the elements of the Vector - in other words, it's the number of times that a particular word appears in the product description.

So you can think of it something like this:

**Decision Tree**  
\- IF the word "TV" appears more than 3 times THEN  
-- IF the word "LED" appears more than 2 times THEN  
--- IF the word "HD" appears at least once THEN  
---- Price = $500


With Random Forest, multiple decision trees are created. Each one is trained with a different random subset of the data, and a different random subset of the features. You can see above that we specify 100 trees, which is the default.

Then the Random Forest model simply takes the average of all its trees to product the final result.


In [31]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [36]:
def random_forest_pricer(item):
    features = vectorizer.transform([item.summary])

    return max(0,rf_model.predict(features)[0])

In [37]:
evaluate(random_forest_pricer,test)

  0%|          | 0/200 [00:00<?, ?it/s]

$36 $82 $12 $32 $18 $116 $68 $67 $7 $196 $490 $281 $42 $70 $15 $1 $71 $71 $16 $86 $16 $108 $22 $10 $92 $343 $153 $155 $25 $49 $56 $124 $61 $3 $39 $462 $30 $31 $150 $118 $121 $92 $2 $61 $127 $35 $39 $50 $56 $26 $12 $31 $309 $43 $298 $47 $49 $127 $7 $45 $118 $1 $62 $19 $376 $83 $29 $178 $46 $160 $20 $16 $75 $134 $26 $88 $99 $27 $21 $51 $42 $72 $37 $44 $3 $31 $5 $135 $50 $63 $32 $215 $1 $9 $75 $35 $21 $50 $115 $247 $67 $1 $10 $67 $57 $25 $160 $197 $35 $204 $30 $69 $55 $11 $119 $49 $47 $21 $222 $44 $24 $33 $60 $38 $42 $17 $41 $73 $84 $28 $51 $51 $101 $9 $40 $29 $74 $3 $48 $42 $2 $139 $21 $60 $99 $61 $48 $321 $63 $15 $9 $115 $102 $66 $46 $56 $131 $24 $69 $16 $28 $11 $5 $41 $358 $34 $180 $31 $16 $55 $39 $35 $272 $34 $6 $10 $49 $2 $53 $13 $285 $27 $79 $68 $137 $72 $74 $115 $15 $46 $30 $71 $8 $21 $7 $26 $86 $70 $32 $13 

## Introducing XGBoost

Like Random Forest, XGBoost is also an ensemble model that combines multiple decision trees.

But unlike Random Forest, XGBoost builds one tree after another, with each next tree correcting for errors in the prior trees, using 'gradient descent'.

It's much faster than Random Forest, so we can run it for the full dataset, and it's typically better at generalizing.

**If this import doesn't work, please skip this! It's not required. On a Mac, you might need to do `brew install libomp` in the terminal.**

In [44]:

import xgboost as xgb

In [45]:
xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [46]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [47]:
evaluate(xg_boost, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$64 $76 $29 $3 $89 $141 $7 $84 $56 $56 $583 $248 $79 $151 $7 $5 $60 $10 $22 $12 $3 $89 $9 $77 $193 $224 $37 $30 $51 $23 $77 $96 $2 $1 $32 $260 $29 $59 $146 $77 $160 $80 $62 $20 $132 $85 $41 $46 $37 $19 $10 $74 $280 $9 $131 $32 $54 $218 $20 $4 $61 $10 $45 $23 $432 $75 $11 $28 $121 $123 $11 $46 $116 $113 $58 $179 $144 $20 $51 $124 $6 $105 $123 $37 $7 $53 $61 $126 $101 $179 $28 $227 $35 $20 $26 $63 $83 $22 $167 $159 $71 $30 $3 $43 $57 $78 $89 $188 $31 $114 $36 $39 $76 $53 $82 $27 $75 $26 $207 $58 $45 $105 $126 $23 $107 $25 $15 $150 $15 $25 $56 $101 $94 $39 $56 $63 $75 $44 $12 $31 $4 $56 $14 $63 $77 $44 $1 $190 $30 $27 $19 $144 $21 $50 $192 $106 $89 $10 $53 $10 $17 $9 $4 $10 $372 $7 $159 $50 $8 $25 $15 $19 $306 $38 $22 $31 $39 $5 $19 $21 $281 $186 $86 $40 $109 $88 $50 $62 $89 $26 $48 $44 $5 $47 $28 $36 $48 $21 $71 $33 